# BigBird on MedMCQA
* [BigBird on Hugging Face](https://huggingface.co/docs/transformers/model_doc/big_bird)
* [MedMCQA on Hugging Face](https://huggingface.co/datasets/openlifescienceai/medmcqa)
* Resources
  - [Training and fine-tuning](https://huggingface.co/transformers/v4.2.2/training.html)
  - [Leaderboard](https://medmcqa.github.io/)

## Experiment

1.  Use out-of-the box BigBird on MedMCQA
    - BigBirdForMultipleChoice
    - BigBirdForQA

In [ ]:
!pip install datasets

In [ ]:
import torch

from torch.utils.data import RandomSampler, Dataset, DataLoader
from torch.nn.utils import rnn, clip_grad_norm_
from torch.optim import AdamW

from transformers import AutoTokenizer, BigBirdForMultipleChoice

from datasets import load_dataset
from tqdm import tqdm

from sklearn.metrics import classification_report, precision_score, recall_score, accuracy_score

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

print(f'on device: {device}')

on device: cuda:0


In [ ]:
model_id = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract'

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

### Collate Function

1. Question - **Pulp proper resembles**
2. Options
      - (a) Loose connective tissue
      - (b) Fine nerve fibers and blood vessels
      - (c) Multipotent cells
      - (d) All of the above

3. Tokenized



    **['[CLS] Pulp proper resembles[SEP] Loose connective tissue[SEP]pad',**

    **'[CLS] Pulp proper resembles[SEP] Fine nerve fibers and blood vessels[SEP]',**

    **'[CLS] Pulp proper resembles[SEP] Multipotent cells[SEP]padpad',**

    **'[CLS] Pulp proper resembles[SEP] All of the above[SEP]padpad']**



In [ ]:
class HFDataset(Dataset):

    def __init__(self, hf):

        self.ds = list(hf.map(HFDataset.process_fn,  batched=True, batch_size=8, remove_columns=hf.column_names))

    @staticmethod
    def process_fn(b):

        question, opa, opb, opc, opd, cop = b['question'], b['opa'], b['opb'], b['opc'], b['opd'], b['cop']

        return  {
                'question': question,
                'options':  list(zip(opa, opb, opc, opd)),
                'answer': cop
            }

    def __getitem__(self, index):

        return self.ds[index]

    def __len__(self):

        return len(self.ds)

class MedMCQA(Dataset):

    def __init__(self, dataset_id='openlifescienceai/medmcqa'):
        hf = load_dataset(dataset_id, split='validation').shuffle(seed=42)

        self.val = HFDataset(hf)

    def __getitem__(self, index):
        raise NotImplementedError('Object of type MedMCQA has no index. Use ".train", ".val", or ".test" instead.')

    def __len__(self):
        raise NotImplementedError('Object of type MedMCQA has no len')

    @staticmethod
    def collate_fn(batch):
        qs, ops, ans = [], [], []

        for e in batch:
            qs.extend([e['question']] * 4)
            ops.extend(e['options'])
            ans.append(e['answer'])

        # dynamically truncate to max_model_seq_length and pad to batch_max_seq
        inputs = tokenizer(qs, ops, return_tensors='pt', padding=True, truncation=True).to(device)
        labels = torch.tensor(ans, dtype=torch.long).to(device)

        batch_size = len(batch)
        num_choices = 4
        seq_len = inputs['input_ids'].shape[1]

        return {k: v.view(batch_size, num_choices, seq_len) for k, v in inputs.items()}, labels


medmcqa = MedMCQA()

val_loader = DataLoader(medmcqa.val, batch_size=4, shuffle=False, collate_fn=MedMCQA.collate_fn, sampler=RandomSampler(medmcqa.val))


### Fine-Tune

In [ ]:
from transformers import AutoModelForMultipleChoice

# model = BigBirdForMultipleChoice.from_pretrained(model_id, attention_type='original_full', num_labels=1).to(device)
# BigBirdClinical and with contexts
model = AutoModelForMultipleChoice.from_pretrained(model_id, num_labels=1).to(device)

params = sum(p.numel() for p in model.parameters())
print(f'params: {round(params / (10 ** 9), 4)} B')

model

Some weights of BertForMultipleChoice were not initialized from the model checkpoint at microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


params: 0.1095 B


BertForMultipleChoice(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, ele

In [ ]:
datapoints = list(val_loader)[ : 100]

In [ ]:
# automatic mixed-precision training with freezing the first layers

layers = model.base_model.encoder.layer

num_layers_to_freeze = int(len(layers) * .3)

print(f'freezeing first {num_layers_to_freeze}/{len(layers)} layers')

for p in layers[ : num_layers_to_freeze].parameters():
    p.requires_grad = False

for p in layers[num_layers_to_freeze : ].parameters():
    p.requires_grad = True

for p in model.classifier.parameters():
    p.requires_grad = True

## trainable parameters
# params2train = [n for n, p in model.named_parameters() if p.requires_grad]

scaler = torch.amp.GradScaler('cuda')
optimizer = AdamW(model.parameters(), lr=1e-5)

epochs = 10
num_train_steps = epochs * len(val_loader)


for epoch in range(epochs):

  loss_epoch = 0.0

  for inputs, labels in tqdm(datapoints, desc='fine-tuning in process'):

      # forward pass
        ## inputs.input_ids = (batch_size, num_choices, seq_length)
        ## labels = (batch_size, )

      optimizer.zero_grad(set_to_none=True)

      with torch.autocast(device_type=device, dtype=torch.float16):

            J = model(**inputs, labels=labels)

            logits, loss = J.logits, J.loss
            loss_epoch += loss.item()

      # backward pass
      scaler.scale(loss).backward()

      scaler.unscale_(optimizer)

      clip_grad_norm_(model.parameters(), max_norm=1.0)

      scaler.step(optimizer)

      scaler.update()


  print(f'\n\nepoch: {epoch + 1} loss: {loss_epoch / len(datapoints)}\n\n')
  print('running per epoch evaluation')


  Y_pred, Y_gold = [], []
  for inputs, labels in tqdm(datapoints, desc='evaluation'):
          J = model(**inputs)
          pred = torch.argmax(J.logits, dim=1)

          Y_pred.extend(pred.tolist())
          Y_gold.extend(labels.tolist())

  # print(classification_report(Y_gold, Y_pred))

  p_macro = precision_score(Y_gold, Y_pred, average='macro')
  r_macro = recall_score(Y_gold, Y_pred, average='macro')
  acc = accuracy_score(Y_gold, Y_pred)

  f1_macro = 2 * p_macro * r_macro / (p_macro + r_macro) if p_macro + r_macro > 0.0 else 0.0

  evald = {'accuracy': acc, 'P-macro': p_macro, 'R-macro': r_macro, 'F1-macro': f1_macro}

  print(*evald.items(), sep='\n')

torch.cuda.empty_cache()

freezeing first 3/12 layers


fine-tuning in process: 100%|██████████| 100/100 [00:06<00:00, 14.66it/s]




epoch: 1 loss: 1.32002685546875


running per epoch evaluation


evaluation: 100%|██████████| 100/100 [00:01<00:00, 56.99it/s]


('accuracy', 0.8025)
('P-macro', 0.797027869118662)
('R-macro', 0.8067049316637482)
('F1-macro', 0.8018372042746651)


...

('accuracy', 1.0)
('P-macro', 1.0)
('R-macro', 1.0)
('F1-macro', 1.0)


fine-tuning in process: 100%|██████████| 100/100 [00:06<00:00, 14.66it/s]




epoch: 10 loss: 0.004260489642620087


running per epoch evaluation


evaluation: 100%|██████████| 100/100 [00:01<00:00, 58.17it/s]

('accuracy', 1.0)
('P-macro', 1.0)
('R-macro', 1.0)
('F1-macro', 1.0)
